# Model Drift: When Yesterday's Model Meets Today's Data

## One-hour machine learning case study

A customer-churn model performed well when it was launched. Six months later, complaints increase and recall falls.

The code did not change. The world did.

This notebook explores data drift, concept drift, delayed labels, monitoring, retraining triggers, and the danger of assuming that deployment is the end of the modeling process.

> **Central question:** How do we know when a model is no longer suitable for the environment in which it operates?

## Learning objectives

Students will:

- distinguish data drift from concept drift;
- compare training-period and current feature distributions;
- track model performance over time;
- understand why stable code does not imply stable behavior;
- design monitoring metrics and retraining triggers;
- recognize the challenge of delayed ground-truth labels;
- use AI to critique a monitoring plan;
- recommend a response that includes rollback and human oversight.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, precision_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)

def generate_period(period, n, usage_shift=0, support_shift=0, coefficient_shift=0):
    monthly_usage = np.clip(rng.normal(65 + usage_shift, 18, n), 0, 120)
    support_tickets = rng.poisson(1.5 + support_shift, n)
    tenure_months = np.clip(rng.gamma(3.5, 9, n), 1, 120)
    payment_failures = rng.binomial(3, 0.08, n)

    logit = (
        -1.7
        - 0.025 * monthly_usage
        + (0.55 + coefficient_shift) * support_tickets
        - 0.012 * tenure_months
        + 0.9 * payment_failures
    )
    prob = 1 / (1 + np.exp(-logit))
    churned = rng.binomial(1, prob)

    return pd.DataFrame({
        "period": period,
        "monthly_usage": monthly_usage,
        "support_tickets": support_tickets,
        "tenure_months": tenure_months,
        "payment_failures": payment_failures,
        "churned": churned
    })

train_period = generate_period("Training", 4000)
month_1 = generate_period("Month 1", 1200, usage_shift=-2, support_shift=0.1)
month_2 = generate_period("Month 2", 1200, usage_shift=-7, support_shift=0.4, coefficient_shift=0.15)
month_3 = generate_period("Month 3", 1200, usage_shift=-14, support_shift=0.8, coefficient_shift=0.35)

all_periods = pd.concat([train_period, month_1, month_2, month_3], ignore_index=True)
all_periods.groupby("period").mean(numeric_only=True).round(2)

# Part 1 — Train the original model

In [ ]:
features = [
    "monthly_usage",
    "support_tickets",
    "tenure_months",
    "payment_failures"
]

X_train, X_test, y_train, y_test = train_test_split(
    train_period[features],
    train_period["churned"],
    test_size=0.25,
    random_state=42,
    stratify=train_period["churned"]
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

test_pred = model.predict(X_test)

pd.Series({
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred, zero_division=0),
    "Recall": recall_score(y_test, test_pred, zero_division=0)
}).to_frame("Launch evaluation").style.format("{:.1%}")

# Part 2 — Evaluate later periods

The model and threshold remain unchanged.

In [ ]:
def evaluate_period(data):
    pred = model.predict(data[features])
    return pd.Series({
        "Customers": len(data),
        "Churn rate": data["churned"].mean(),
        "Accuracy": accuracy_score(data["churned"], pred),
        "Precision": precision_score(data["churned"], pred, zero_division=0),
        "Recall": recall_score(data["churned"], pred, zero_division=0)
    })

performance = pd.DataFrame({
    "Training holdout": pd.Series({
        "Customers": len(y_test),
        "Churn rate": y_test.mean(),
        "Accuracy": accuracy_score(y_test, test_pred),
        "Precision": precision_score(y_test, test_pred, zero_division=0),
        "Recall": recall_score(y_test, test_pred, zero_division=0)
    }),
    "Month 1": evaluate_period(month_1),
    "Month 2": evaluate_period(month_2),
    "Month 3": evaluate_period(month_3)
}).T

performance.style.format({
    "Churn rate": "{:.1%}",
    "Accuracy": "{:.1%}",
    "Precision": "{:.1%}",
    "Recall": "{:.1%}"
})

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(performance.index, performance["Recall"], marker="o")
plt.ylim(0, 1)
plt.ylabel("Recall")
plt.xlabel("Evaluation period")
plt.title("Churn-model recall over time")
plt.grid(True, alpha=0.3)
plt.show()

## Interpretation

1. Which metric deteriorated most?
2. Could accuracy remain acceptable while recall becomes unacceptable?
3. What business consequences follow from missed churners?
4. What changed even though the code remained the same?

# Part 3 — Detect data drift

Compare feature distributions over time.

In [ ]:
feature_means = (
    all_periods.groupby("period")[features]
    .mean()
    .loc[["Training", "Month 1", "Month 2", "Month 3"]]
)

feature_means.round(2)

In [ ]:
plt.figure(figsize=(9, 5))
for period, data in all_periods.groupby("period"):
    plt.hist(
        data["monthly_usage"],
        bins=25,
        alpha=0.35,
        density=True,
        label=period
    )
plt.xlabel("Monthly usage")
plt.ylabel("Density")
plt.title("Monthly-usage distribution by period")
plt.legend()
plt.show()

## Data drift versus concept drift

**Data drift:** The distribution of inputs changes.

Example: customers now use the product less often.

**Concept drift:** The relationship between inputs and the target changes.

Example: support tickets become a stronger warning sign of churn after a service redesign.

A system may experience either or both.

# Part 4 — Monitoring without immediate labels

Ground-truth churn labels may arrive weeks or months later.

Before labels arrive, teams can monitor:

- feature distributions;
- missing-value rates;
- prediction-score distributions;
- percentage of customers flagged;
- system errors and latency;
- business process changes.

These are warning signals, not substitutes for later performance evaluation.

# Part 5 — Design retraining and rollback rules

A monitoring plan should specify:

- which metrics are tracked;
- acceptable ranges;
- alert thresholds;
- who investigates;
- what happens during investigation;
- when retraining is allowed;
- when rollback is required;
- how the new model is validated.

Create a plan for this churn model.

### Monitoring plan

**Feature-drift metrics:**  

**Performance metrics:**  

**Business metrics:**  

**Warning threshold:**  

**Critical threshold:**  

**Temporary safeguard:**  

**Retraining requirement:**  

**Rollback condition:**

## AI as an operations reviewer

Use one prompt:

> Review my drift-monitoring plan and identify missing failure modes.

> What can be monitored before true churn labels become available?

> Challenge my proposed retraining trigger.

> What risks arise from automatically retraining without human review?

Evaluate:

**Useful warning:**  

**Unsupported assumption:**  

**Missing operational metric:**  

**Change to the plan:**

# Part 6 — Responsible response

Possible responses to drift include:

- investigate the data pipeline;
- adjust the decision threshold temporarily;
- retrain with recent labeled data;
- add new relevant features;
- roll back to a safer rule-based process;
- increase human review;
- pause automated actions.

The correct response depends on the cause. Retraining is not a universal solution.

# Transfer task — Loan default

A loan-default model was trained before a major economic change.

Current applicants have:

- lower average income;
- higher debt-to-income ratios;
- different employment patterns.

Labels for default will not be complete for 12 months.

Answer:

1. What drift can be detected immediately?
2. What cannot yet be measured reliably?
3. What temporary safeguards are appropriate?
4. When should the model be paused?
5. Why might retraining on a small amount of recent data be risky?

# Exit reflection

- Deployment is not the end because …
- Data drift means …
- Concept drift means …
- Monitoring without labels can reveal …
- Retraining should occur only when …

# Instructor checklist

- [ ] Students compared performance over time.
- [ ] Students distinguished data and concept drift.
- [ ] Students monitored feature distributions.
- [ ] Students recognized delayed labels.
- [ ] Students designed alerts, safeguards, and rollback rules.
- [ ] AI audited the monitoring plan.
- [ ] Students transferred the idea to a financial setting.

# Closing principle

A model is a living component inside a changing system.

Trust must be earned repeatedly through monitoring, evaluation, investigation, and responsible intervention.